In [ ]:
import os
import time

import pycuda.autoinit
from utils.yolo_classes import get_cls_dict
from utils.display import open_window, set_display, show_fps
from utils.visualization import BBoxVisualization
from utils.yolo import TRT_YOLO

trt_yolo = TRT_YOLO("yolov4-tiny-416", (416, 416), 4)

In [ ]:
import cv2
import ipywidgets.widgets as widgets
from IPython.display import display

# 1. 讀取影像
img = cv2.imread('1.jpg')

# 2. 進行模型辨識
boxes, confs, clss = trt_yolo.detect(img, 0.3)

# 3. 跑迴圈將辨識結果畫在影像上
for i in range(len(clss)):
    cv2.rectangle(img, (boxes[i][0], boxes[i][1]), (boxes[i][2], boxes[i][3]), (255, 0, 0), 2)

# 將 OpenCV 的影像編碼成 JPEG 格式
_, encoded_image = cv2.imencode('.jpg', img)

# 建立一個 Image 元件並顯示
image_widget = widgets.Image(value=encoded_image.tobytes(), format='jpg', width=400)
display(image_widget)

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

In [ ]:
import traitlets
from IPython.display import display
import ipywidgets.widgets as widgets
from jetbot import Camera, bgr8_to_jpeg
from jetbot import Robot

# 只有使用 USBCamera 的需要這幾行
# from jetcam.usb_camera import USBCamera
# camera = USBCamera(capture_device=0)
# camera.running = True

# 一般 camera (CSI 介面) 的用這一行
camera = Camera.instance(width=416, height=416)

image = widgets.Image(format='jpeg', width=224, height=224)
camera_link = traitlets.dlink((camera, 'value'), (image, 'value'), transform=bgr8_to_jpeg)

robot = Robot()

In [ ]:
import numpy as np

def getROI(img, vertices):
    mask = np.zeros_like(img) # 建立遮罩
    cv2.fillPoly(mask, vertices, (255,)) # 把多邊形填滿白色
    masked_image = cv2.bitwise_and(img, mask)
    return masked_image # 找要跟蹤的點

def make_points(image, average):
    slope, y_int = average
    y1 = image.shape[0]
    y2 = int(y1 * (1/6))
    x1 = int((y1 - y_int) // slope)
    x2 = int((y2 - y_int) // slope)
    return np.array([x1, y1, x2, y2])

In [ ]:
def average(image, lines):
    if lines is None: # 將多條線中畫出左右各一條線
        return
    left = []
    right = []
    for line in lines:
        x1, y1, x2, y2 = line.reshape(4)
        parameters = np.polyfit((x1, x2), (y1, y2), 1)
        slope = parameters[0]
        y_int = parameters[1]
        if slope < 0:
            left.append((slope, y_int))
        else:
            right.append((slope, y_int))

    right_avg = np.average(right, axis=0)
    left_avg = np.average(left, axis=0)

    if type(left_avg) is np.float64:
        right_line = make_points(image, right_avg)
        return np.array([right_line,])
    if type(right_avg) is np.float64:
        left_line = make_points(image, left_avg)
        return np.array([left_line,])

    right_line = make_points(image, right_avg)
    left_line = make_points(image, left_avg)
    return np.array([left_line, right_line])

In [ ]:
def getAverageLines(origin_road):
    # 將影像縮放為 640x480
    origin_road = cv2.resize(origin_road, (640, 480))

    # 讀入 RGB 影像轉換為灰階景
    road_gray = cv2.cvtColor(origin_road, cv2.COLOR_BGR2GRAY)

    # 濾波：使用高斯模糊去除雜訊
    road_gray = cv2.GaussianBlur(road_gray, (5, 5), 0)

    # 邊緣檢測：使用 Canny 演算法
    edges = cv2.Canny(road_gray, 3000, 2000, apertureSize=5)

    # 定義感興趣區域 (ROI) 的頂點座標
    ROI_vertices = [(0, 480), (0, 150), (250, 80), (390, 80), (640, 150), (640, 480)]

    # 僅保留 ROI 區域內的邊緣
    edges = getROI(edges, np.array([ROI_vertices], np.int32))

    # 直線偵測：找直線 (使用機率霍夫變換)
    linesp = cv2.HoughLinesP(edges, rho=2, theta=np.pi / 180,
                             threshold=100, minLineLength=40, maxLineGap=5)

    # 回傳平均化後的左右車道線
    return average(origin_road, linesp)

In [ ]:
def modifySpeed(origin_road):
    global speed_left, speed_right, speed

    # 取得處理後的左右車道線平均座標
    averaged_lines = getAverageLines(origin_road)

    if averaged_lines is not None:
        # 計算偵測到的車道線中點的 X 座標
        mid_x = int(np.average(averaged_lines, axis=0)[2])

        # 根據中點位置判斷偏向並調整速度
        if mid_x < 315:
            # 中點偏左，增加右輪速度以向左修正
            speed_left = speed
            speed_right = speed + 0.03
        elif mid_x > 325:
            # 中點偏右，增加左輪速度以向右修正
            speed_left = speed + 0.02
            speed_right = speed
        else:
            # 接近正中央，微調左輪維持穩定
            speed_left = speed + 0.01
            speed_right = speed

In [ ]:
import time
stop_ignore = 0
railway_ignore = 0
count = 0

def getNearest(signs):
    # 如果沒有偵測到任何標誌，回傳預設值 [0, 3]
    if not len(signs):
        return [0, 3]

    t = time.time()

    # 判斷標誌類型與冷卻時間
    # signs[0][1] == 0: 假設是停止標誌
    # signs[0][1] == 1: 假設是平交道標誌
    # signs[0][1] in [2, 3]: 其他標誌（可能不需冷卻）
    if (signs[0][1] == 0 and (t > stop_ignore)) \
        or (signs[0][1] == 1 and (t > railway_ignore)) \
        or (signs[0][1] in [2, 3]):
        return signs[0]

    # 遞迴檢查下一個標誌
    return getNearest(signs[1:]) if len(signs) >= 2 else [0, 3]

In [ ]:
def update(change):
    global robot, count, speed_left, speed_right, stop_ignore, railway_ignore
    ALERT_WIDTH = 50

    img = change['new']
    modifySpeed(img) # 執行車道追蹤與速度修正

    if count < 10:
        count += 1
        return
    else:
        count = 0

    # 偵測路牌
    boxes, confs, clss = trt_yolo.detect(img)

    signs = []
    for box, cls in zip(boxes, clss):
        width = box[2] - box[0]
        signs.append([width, cls])

    # 找出最靠近車子的路牌 (依寬度排序)
    signs.sort(reverse=True, key=lambda x : x[0])
    sign = getNearest(signs)

    t = time.time()

    # 根據標誌類型執行動作
    if sign[1] == 2 and sign[0] > ALERT_WIDTH: # 減速行駛
        print('slow')
        robot.set_motors(speed_left * 0.7, speed_right * 0.7)

    if (sign[1] == 3 and sign[0] > ALERT_WIDTH): # 道路封閉
        print("road close")
        robot.stop()

    elif sign[1] == 0 and sign[0] > ALERT_WIDTH: # 停車再開
        print("stop")
        robot.stop()
        stop_ignore = t + 10
        time.sleep(3)

    elif sign[1] == 1 and sign[0] > (ALERT_WIDTH - 20): # 鐵路平交道
        print("railway")
        robot.stop()
        railway_ignore = t + 10
        time.sleep(5)

    else:
        # 無特殊標誌，按正常車道追蹤速度行駛
        robot.set_motors(speed_left, speed_right)

In [ ]:
import cv2
import ipywidgets.widgets as widgets
from IPython.display import display
import numpy as np

# 1. 驗證鏡頭畫面
img = camera.value

# 2. 偵測路牌標誌
boxes, confs, clss = trt_yolo.detect(img, 0.3)

# 3. 取得車道平均線
averaged_lines = getAverageLines(img)

if averaged_lines is not None:
    for x1, y1, x2, y2 in averaged_lines:
        # 畫出偵測到的車道紅線 (座標縮放：640->224, 480->224)
        cv2.line(img, (int(x1 / 640 * 224), int(y1 / 480 * 224)),
                 (int(x2 / 640 * 224), int(y2 / 480 * 224)), (0, 0, 255), 5)

    # 計算並畫出路徑中點的綠色標記
    mid_x = int(np.average(averaged_lines, axis=0)[2])
    cv2.line(img, (int(mid_x / 640 * 224), 100), (int(mid_x / 640 * 224), 105), (0, 255, 0), 2)
    print('mid: ', mid_x)

# 4. 畫出路牌的辨識框
for i in range(len(clss)):
    cv2.rectangle(img, (boxes[i][0], boxes[i][1]), (boxes[i][2], boxes[i][3]), (0, 0, 255), 1)

# 5. 【替換掉 imshow】使用 ipywidgets 顯示
# 因為 widgets.Image 會直接處理編碼，這裡不需要手動將 BGR 轉 RGB 也能正確顯示
_, encoded_image = cv2.imencode('.jpg', img)
image_widget = widgets.Image(value=encoded_image.tobytes(), format='jpg', width=224, height=224)

display(image_widget)

mid:  315


Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

In [ ]:
speed = speed_left = speed_right = 0.35

# 車子起步時的速度（給予較大推力以克服靜摩擦力）
robot.set_motors(0.37, 0.41)
time.sleep(0.3)

# 設定回預設巡航速度
robot.set_motors(speed_left, speed_right)

# 進入無窮迴圈，持續讀取影像並更新動作
while(1):
    update({'new' : camera.value})

/usr/local/lib/python3.6/dist-packages/numpy/lib/function_base.py:380: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis)
/usr/local/lib/python3.6/dist-packages/numpy/core/_methods.py:170: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packa

stop


/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly cond

stop
railway
road close


/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned


...

slow
railway


/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned


railway


/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly conditioned
/usr/local/lib/python3.6/dist-packages/ipykernel_launcher.py:25: RankWarning: Polyfit may be poorly cond

stop


In [ ]:
speed = speed_left = speed_right = 0
robot.stop()

In [ ]:
camera_link.unlink()

In [ ]:
camera.stop()